# Clase 8 · De leer pandas a contar una historia con datos
### Análisis y visualización de datos · FLACSO Ecuador · Diversa

**Pregunta guía:** ¿Cómo se distribuye el diálogo entre personajes clasificados como mujeres y hombres en una colección de guiones cinematográficos?

Usaremos datos publicados junto con [la historia de The Pudding sobre diálogo en cine](https://pudding.cool/2017/03/film-dialogue/) y el [repositorio de datos de Polygraph](https://github.com/matthewfdaniels/scripts). [Polygraph](https://polygraph.cool/) es el estudio del mismo equipo. Nos inspiramos en su forma de **plantear una pregunta, revelar datos por etapas y explicar los límites**; este cuaderno es una actividad docente propia, no una reproducción del diseño ni de los resultados de la historia.

**Ruta (aprox. 2 horas):** 25 min lectura de código · 25 min exploración y calidad · 30 min análisis · 25 min gráficos y narrativa · 15 min reto y cierre. Ejecuta las celdas en orden. En Colab: *Entorno de ejecución → Ejecutar todas*.

## 1. Guía de lectura: Python y pandas sin memorizar recetas

**Un objeto es algo que tenemos en memoria**: un número (`int`/`float`), texto (`str`), lista (`list`), tabla (`DataFrame`) o columna (`Series`). Una variable es el nombre con que guardamos ese objeto.

```python
resultado = objeto.metodo(argumentos)
```

Se lee: asigna a `resultado` lo que devuelve un método del objeto. `=` **asigna**; `==` **compara**. `.` permite acceder a un atributo o método. `()` llama a un método o función y puede recibir argumentos. `[]` selecciona columnas o filtra filas. Las comillas delimitan texto (incluidos nombres de columna). `:` inicia bloques como `if` o `for`, con sangría.

| Tipo | Ejemplo | ¿Cuándo usarlo? | Devuelve |
|---|---|---|---|
| Variable | `tabla = ...` | Guardar un resultado | El objeto queda disponible como `tabla` |
| Atributo | `tabla.shape`, `tabla.columns` | Consultar una característica | Dimensiones, nombres |
| Método | `tabla.head(3)`, `tabla.info()` | Pedir al objeto que haga algo | Depende del método; `info()` imprime y devuelve `None` |
| Función | `len(tabla)`, `print(tabla)` | Operación general de Python | Número o salida en pantalla |
| Selección | `tabla["gender"]` | Trabajar con una columna | `Series` |
| Filtro | `tabla[tabla["gender"] == "f"]` | Quedarse con ciertas filas | `DataFrame` |

**La clave para encadenar:** lee de izquierda a derecha; cada paso recibe el resultado del paso anterior. Pregunta siempre: «¿Tengo una tabla, una columna, un grupo, un número o una gráfica?»

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

mini = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
print("Tipo de mini:", type(mini).__name__)
print("Atributo shape:", mini.shape)
print("Tipo de una columna:", type(mini["valor"]).__name__)
print("Resultado del método mean:", mini["valor"].mean())
mini

### Práctica de lectura antes de ejecutar

Predice y explica: ¿qué devuelven `mini["valor"] > 15`, `mini[mini["valor"] > 15]` y `mini.groupby("grupo")["valor"].mean()`? ¿Cuál es la diferencia entre `mini.shape` y `mini.head()`?

In [ ]:
print("Comparación → booleanos:")
print(mini["valor"] > 15)
print("Filtro → tabla:")
display(mini[mini["valor"] > 15])
print("Agrupación → una media por grupo:")
display(mini.groupby("grupo")["valor"].mean())

## 2. Mapa de operaciones: primero la pregunta, luego el método

| Pregunta | Operación | Ejemplo con `tabla` | Resultado |
|---|---|---|---|
| ¿Cómo luce? | `head(n)`, `tail(n)`, `sample(n)` | `tabla.head(3)` | Filas |
| ¿Qué estructura tiene? | `shape`, `columns`, `dtypes`, `info()` | `tabla.info()` | Diagnóstico |
| ¿Qué parte necesito? | `[]`, `loc[]` | `tabla[["gender", "words"]]` | Columna o tabla |
| ¿Cuántos valores faltan? | `isna().sum()` | `tabla.isna().sum()` | Conteos por columna |
| ¿Qué categorías hay? | `value_counts()`, `unique()`, `nunique()` | `tabla["gender"].value_counts()` | Frecuencias o valores |
| ¿Cómo se distribuye una medida? | `describe()`, `min()`, `max()`, `mean()`, `median()` | `tabla["words"].median()` | Resumen o número |
| ¿Qué filas cumplen una regla? | Comparación + máscara, `isin()` | `tabla[tabla["gender"] == "f"]` | Filas filtradas |
| ¿Qué está arriba? | `sort_values()`, `head()` | `tabla.sort_values("words", ascending=False).head(5)` | Tabla ordenada |
| ¿Qué cambia entre grupos? | `groupby().size()`, `count()`, `sum()`, `mean()`, `median()` | `tabla.groupby("gender")["words"].sum()` | Resumen por grupo |
| ¿Necesito varias medidas? | `groupby().agg()` | `tabla.groupby("gender").agg(...)` | Tabla resumen |
| ¿Quiero combinar tablas? | `merge(..., on=..., validate=...)` | `tabla.merge(otra, on="script_id")` | Tabla ampliada |
| ¿Quiero dibujar? | `plot.bar()`, `plot.hist()`, `plt.scatter()` | `tabla["words"].plot.hist()` | Gráfico |
| ¿Quiero guardar? | `to_csv()` | `tabla.to_csv("resultado.csv", index=False)` | Archivo |

**Detalles útiles:** `size()` cuenta filas, incluso si otra columna está vacía; `count()` cuenta valores no vacíos de la columna seleccionada. `sum()` es un total y `mean()` un promedio; contestan preguntas distintas. `reset_index()` convierte las etiquetas de grupo en una columna ordinaria. Algunos métodos, como `info()` y `to_csv()`, producen un efecto y no una nueva tabla.

### Cómo leer una cadena larga

```python
resultado = (tabla.groupby("gender")["words"]
                  .sum()
                  .sort_values(ascending=False)
                  .round(0))
```

1. `tabla` es un `DataFrame`.
2. `groupby("gender")` define los grupos.
3. `["words"]` elige la medida.
4. `sum()` produce un total por grupo (`Series`).
5. `sort_values(...)` lo ordena; `round(0)` lo redondea.
6. `resultado` guarda el objeto final. Los paréntesis exteriores permiten escribir la expresión en varias líneas.

**Argumentos:** `ascending=False` significa «orden descendente». Un argumento puede cambiar cómo funciona un método. Pide ayuda en Colab con `tabla.sort_values?` o usa `help(pd.DataFrame.sort_values)`.

## 3. Datos y procedencia

The Pudding publicó la historia y enlazó su repositorio. Usaremos `character_list5.csv` (personajes, número de palabras, clasificación `f`/`m` y edad) y `meta_data7.csv` (película y año). Son **guiones seleccionados**, con criterios y errores posibles; no equivalen al conjunto de películas estrenadas. La fuente describe un umbral de al menos 100 palabras por personaje mapeado y advierte diferencias entre guion y película final.

Las categorías `f` y `m` proceden del método de clasificación original; **no identifican la identidad de género de las personas ni representan todas las identidades**. Analizaremos palabras asignadas a personajes bajo esas etiquetas históricas. No inferiremos causas ni experiencias de actores o espectadores.

Fuentes: [historia y método](https://pudding.cool/2017/03/film-dialogue/) · [README y datos originales](https://github.com/matthewfdaniels/scripts). La descarga requiere conexión; si falla, descarga ambos CSV del repositorio y súbelos a Colab.

In [ ]:
BASE = "https://raw.githubusercontent.com/matthewfdaniels/scripts/graphs/"
try:
    personajes = pd.read_csv(BASE + "character_list5.csv", encoding="latin-1")
    peliculas = pd.read_csv(BASE + "meta_data7.csv", encoding="latin-1")
except Exception as error:
    print("Falló la descarga automática:", error)
    print("Sube character_list5.csv y meta_data7.csv desde el panel Archivos de Colab.")
    personajes = pd.read_csv("character_list5.csv", encoding="latin-1")
    peliculas = pd.read_csv("meta_data7.csv", encoding="latin-1")
print("Personajes:", personajes.shape, "Películas:", peliculas.shape)
display(personajes.head(3), peliculas[["script_id", "title", "year"]].head(3))

### Pausa: leer y comprobar la estructura

¿Qué representa una fila de `personajes`? ¿Y una de `peliculas`? `script_id` permite conectarlas. Antes de hacer gráficos, observa columnas, tipos y faltantes.

In [ ]:
print("COLUMNAS DE PERSONAJES")
print(personajes.columns.tolist())
personajes.info()
print("\nFALTANTES")
display(personajes.isna().sum())
print("CATEGORÍAS ORIGINALES")
display(personajes["gender"].value_counts(dropna=False))
print("PALABRAS POR PERSONAJE")
display(personajes["words"].describe().round(1))

**Pregunta para conversar:** `age` tiene valores faltantes. ¿Podemos describir edades de *todos* los personajes a partir de las edades disponibles? ¿Qué información necesitaríamos para evaluar ese sesgo? No rellenaremos edades sin justificación.

## 4. Unir tablas, filtrar y verificar

Una película tiene varios personajes: es una unión **muchos a uno** por `script_id`. Pedimos a pandas verificarlo. Conservamos las etiquetas `f` y `m`; contamos antes las filas con otra etiqueta para hacer visible la decisión.

In [ ]:
print("Etiquetas no incluidas:", (~personajes["gender"].isin(["f", "m"])).sum())
print("Identificadores de películas duplicados:", peliculas["script_id"].duplicated().sum())

tabla = personajes.merge(
    peliculas[["script_id", "title", "year"]],
    on="script_id", how="left", validate="many_to_one"
)
assert tabla["title"].notna().all(), "Hay personajes sin película asociada"
analisis = tabla[tabla["gender"].isin(["f", "m"])].copy()
print("Filas para el análisis:", len(analisis), "Películas:", analisis["script_id"].nunique())
display(analisis.head(3))

**Desarma esta línea:** `tabla[tabla["gender"].isin(["f", "m"])].copy()`. Primero obtén la columna; luego la máscara de `True`/`False`; después las filas. `.copy()` crea una tabla independiente para el análisis.

In [ ]:
mascara = tabla["gender"].isin(["f", "m"])
print("Primeros valores de la máscara:", mascara.head().tolist())
print("Filas seleccionadas:", mascara.sum())

print("Cinco personajes con más palabras registradas:")
display(analisis[["title", "imdb_character_name", "gender", "words"]]
        .sort_values("words", ascending=False).head(5))

## 5. Primera escena: ¿qué se contó?

Una gráfica necesita una **unidad de análisis**. Aquí cada fila representa un personaje mapeado; `words` es el número de palabras atribuidas en el guion. El número de personajes y el número de palabras no miden lo mismo.

In [ ]:
resumen_etiqueta = analisis.groupby("gender").agg(
    personajes=("words", "size"),
    palabras=("words", "sum"),
    mediana_por_personaje=("words", "median"),
    promedio_por_personaje=("words", "mean")
).rename(index={"f": "Clasificación f", "m": "Clasificación m"})
display(resumen_etiqueta.round(1))

**Narración 1 (completa después de mirar la tabla):** «Entre los personajes incluidos, la clasificación ___ reúne ___ personajes y ___ palabras. El total no indica por sí solo cuánto habla un personaje típico; para eso observamos la mediana». ¿Por qué promedio y mediana pueden diferir?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
resumen_etiqueta["palabras"].div(1_000_000).plot.barh(ax=ax, color=["#D26A48", "#334E68"])
ax.set_title("Palabras registradas por clasificación del personaje", loc="left", weight="bold")
ax.set_xlabel("Millones de palabras en los guiones incluidos")
ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Contrapunto: el denominador importa

Un total de palabras de toda la colección mezcla películas y personajes. Para comparar películas, primero calcularemos **la proporción de palabras clasificadas como `f` dentro de cada guion**: palabras `f` / (`f` + `m`). Esta proporción solo utiliza las palabras de personajes incluidos en el archivo, no todo el diálogo de la película.

In [ ]:
por_pelicula = (analisis.groupby(["script_id", "title", "year", "gender"])["words"]
                .sum().unstack(fill_value=0).reset_index())
por_pelicula["total_f_m"] = por_pelicula["f"] + por_pelicula["m"]
por_pelicula["pct_f"] = 100 * por_pelicula["f"] / por_pelicula["total_f_m"]
assert len(por_pelicula) == analisis["script_id"].nunique()
assert por_pelicula["pct_f"].between(0, 100).all()
print("Una fila por película:", por_pelicula.shape)
display(por_pelicula[["title", "year", "f", "m", "pct_f"]].head())
display(por_pelicula["pct_f"].describe().round(1))

**Método nuevo, idea conocida:** `unstack(fill_value=0)` mueve las categorías `f`/`m` del índice a dos columnas. Antes de esa operación teníamos varias filas por película; después, una fila por película. `fill_value=0` significa que, si en el archivo no hay personajes incluidos de una categoría para un guion, esa suma es cero. Comprueba qué contiene `por_pelicula` antes de continuar.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(por_pelicula["pct_f"], bins=range(0, 111, 10), color="#704C88", edgecolor="white")
ax.axvline(50, color="#222", linestyle="--", linewidth=1.5, label="50 %")
ax.set(title="No todas las películas reparten igual el diálogo clasificado",
       xlabel="Porcentaje de palabras de personajes clasificados como f, por guion",
       ylabel="Número de guiones")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print("Mediana entre guiones (%):", round(por_pelicula["pct_f"].median(), 1))
print("Guiones con más de 50 % en f:", (por_pelicula["pct_f"] > 50).sum(), "de", len(por_pelicula))

**Narración 2:** «En vez de un solo total para toda la colección, cada guion ocupa ahora un lugar en la distribución. El eje horizontal expresa porcentaje y el vertical cuenta guiones». ¿Cuántos se ubican por encima de 50 %? La **mediana de porcentajes entre películas** no es lo mismo que el **porcentaje obtenido al sumar todas las palabras**: el segundo da más peso a guiones con más palabras registradas.

## 6. Segunda escena: comparar décadas sin ocultar el tamaño de muestra

Una línea temporal puede sugerir cambios históricos. Este archivo reúne guiones seleccionados de manera desigual; calcularemos el número de películas por década para no presentar la muestra como si fuera una serie representativa de la industria.

In [ ]:
por_pelicula["decada"] = (por_pelicula["year"] // 10) * 10
por_decada = por_pelicula.groupby("decada").agg(
    guiones=("script_id", "size"),
    mediana_pct_f=("pct_f", "median"),
    promedio_pct_f=("pct_f", "mean")
).reset_index()
display(por_decada.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(por_decada["decada"], por_decada["mediana_pct_f"], marker="o", color="#704C88", linewidth=2)
for _, fila in por_decada.iterrows():
    ax.annotate(f'n={int(fila["guiones"])}',
                (fila["decada"], fila["mediana_pct_f"]),
                xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8)
ax.set(title="Mediana por década en los guiones incluidos",
       xlabel="Década de estreno", ylabel="Mediana del porcentaje de palabras f por guion (%)")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Narración 3:** Describe lo que ocurre **en esta selección de guiones**. Lee las etiquetas `n`: ¿hay décadas con pocos casos? No atribuyas una subida o bajada a una causa; ni digas que el gráfico representa la evolución de todo Hollywood. ¿Cambiaría la historia si usáramos el promedio en lugar de la mediana?

## 7. Miniensayo visual: escribir junto a los gráficos

The Pudding suele organizar una pregunta en escenas: **gancho → unidad de análisis → comparación → sorpresa o matiz → límites**. En Colab, las celdas Markdown cumplen la función de los párrafos y las celdas de código producen las figuras. Una gráfica acompaña una afirmación concreta; no hace falta que cada paso tenga un gráfico.

**Plantilla para tu propia historia (edita esta celda o crea una nueva):**

1. **Título con pregunta:** ¿________?
2. **Qué datos tenemos:** ________ (fuente, periodo, unidad, selección).
3. **Primer hallazgo:** ________ (número o comparación verificable y gráfico 1).
4. **Un matiz:** ________ (otra medida, subgrupo o contraejemplo y gráfico 2).
5. **Límite:** ________ (faltantes, método o población que queda fuera).
6. **Cierre:** «Estos datos permiten afirmar ________; aún no permiten concluir ________».

Un buen titular puede ser preciso sin resolver toda la historia en una frase. Escribe unidades y denominadores en los ejes; atribuye la fuente y describe las decisiones de filtrado.

## 8. Reto individual (elige uno)

**A. Sintaxis y método:** calcula `median()`, `mean()` y `max()` de `words` por clasificación. Explica qué tipo de objeto devuelve cada paso y por qué la media podría superar a la mediana.

**B. Filtro y comparación:** elige una década con suficientes guiones; filtra `por_pelicula`, calcula cuántos guiones hay y qué porcentaje supera 50 % de palabras `f`. Compara con otra década, indicando los dos tamaños de muestra.

**C. Tu propia escena:** selecciona tres películas de la tabla `por_pelicula` (por un criterio explícito, no solo extremos), dibuja barras de `pct_f` y redacta un párrafo con denominador y una limitación. Evita generalizar desde tres casos.

**Entrega:** deja una celda Markdown con (1) pregunta, (2) código explicado en lenguaje natural, (3) gráfico o tabla, (4) hallazgo, (5) limitación.

In [ ]:
# Espacio para tu reto. Ejemplo de punto de partida:
# seleccion = por_pelicula[por_pelicula["decada"] == 2000]
# print(len(seleccion))
# display(seleccion["pct_f"].describe())

## 9. Comprobación final

- [ ] Distingo atributo (`shape`) de método (`head()`) y de función (`len()`).
- [ ] Sé leer `df["columna"]`, `df[condición]` y una cadena con métodos.
- [ ] Sé qué representa cada fila antes y después de `groupby` o `merge`.
- [ ] Elijo `size`, `count`, `sum`, `mean` o `median` según mi pregunta.
- [ ] Mis gráficos tienen título, ejes, unidades y un denominador entendible.
- [ ] Mi narrativa incluye fuente, decisiones y límites.

**Referencias:** [The Pudding, historia y método](https://pudding.cool/2017/03/film-dialogue/) · [Polygraph, datos y README](https://github.com/matthewfdaniels/scripts) · [sitio de Polygraph](https://polygraph.cool/).